In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, MaxPooling2D, UpSampling2D
from tensorflow.keras.models import Model
from tensorflow.keras import Input
from tensorflow.keras.optimizers import Adam

D:\Anaconda\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
# Load preprocessed data
X_train = np.load("D:/PomegranateGuard/dataset/X_train_autoencoder.npy")
X_val   = np.load("D:/PomegranateGuard/dataset/X_val_autoencoder.npy")

print("Train Shape:", X_train.shape)
print("Validation Shape:", X_val.shape)

Train Shape: (3938, 224, 224, 3)
Validation Shape: (844, 224, 224, 3)


In [3]:
input_img = Input(shape=(224,224,3))

# ---------------- Encoder ----------------
x = Conv2D(32, (3,3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2,2), padding='same')(x)

x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = MaxPooling2D((2,2), padding='same')(x)

x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
encoded = MaxPooling2D((2,2), padding='same')(x)

# ---------------- Decoder ----------------
x = Conv2D(128, (3,3), activation='relu', padding='same')(encoded)
x = UpSampling2D((2,2))(x)

x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = UpSampling2D((2,2))(x)

x = Conv2D(32, (3,3), activation='relu', padding='same')(x)
x = UpSampling2D((2,2))(x)

decoded = Conv2D(3, (3,3), activation='sigmoid', padding='same')(x)

autoencoder = Model(input_img, decoded)
autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape          ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ conv2d (Conv2D)               │ (None, 224, 224, 32)  │          896 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ max_pooling2d (MaxPooling2D)  │ (None, 112, 112, 32)  │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ conv2d_1 (Conv2D)             │ (None, 112, 112, 64)  │       18,496 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ max_pooling2d_1               │ (None, 56, 56, 64)    │            0 │
│ (MaxPooling2D)                │                       │              │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ conv2d_2 (Conv2D)             │ (None, 56, 56, 128)   │       73,856 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ max_pooling2d_2               │ (None, 28, 28, 128)   │            0 │
│ (MaxPooling2D)                │                       │              │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ conv2d_3 (Conv2D)             │ (None, 28, 28, 128)   │      147,584 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ up_sampling2d (UpSampling2D)  │ (None, 56, 56, 128)   │            0 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ conv2d_4 (Conv2D)             │ (None, 56, 56, 64)    │       73,792 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ up_sampling2d_1               │ (None, 112, 112, 64)  │            0 │
│ (UpSampling2D)                │                       │              │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ conv2d_5 (Conv2D)             │ (None, 112, 112, 32)  │       18,464 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ up_sampling2d_2               │ (None, 224, 224, 32)  │            0 │
│ (UpSampling2D)                │                       │              │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ conv2d_6 (Conv2D)             │ (None, 224, 224, 3)   │          867 │
└───────────────────────────────┴───────────────────────┴──────────────┘

 Total params: 333,955 (1.27 MB)

 Trainable params: 333,955 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
history = autoencoder.fit(
    X_train, X_train,
    epochs=15,
    batch_size=16,
    validation_data=(X_val, X_val),
    verbose=1
)

Epoch 1/15
247/247 ━━━━━━━━━━━━━━━━━━━━ 165s 652ms/step - loss: 0.0076 - val_loss: 3.4407e-06
Epoch 2/15
247/247 ━━━━━━━━━━━━━━━━━━━━ 142s 576ms/step - loss: 3.4220e-06 - val_loss: 3.4407e-06
Epoch 3/15
247/247 ━━━━━━━━━━━━━━━━━━━━ 154s 623ms/step - loss: 3.4220e-06 - val_loss: 3.4407e-06
Epoch 4/15
247/247 ━━━━━━━━━━━━━━━━━━━━ 150s 606ms/step - loss: 3.4220e-06 - val_loss: 3.4407e-06
Epoch 5/15
247/247 ━━━━━━━━━━━━━━━━━━━━ 155s 628ms/step - loss: 3.4220e-06 - val_loss: 3.4407e-06
Epoch 6/15
247/247 ━━━━━━━━━━━━━━━━━━━━ 156s 631ms/step - loss: 3.4220e-06 - val_loss: 3.4407e-06
Epoch 7/15
247/247 ━━━━━━━━━━━━━━━━━━━━ 155s 627ms/step - loss: 3.4220e-06 - val_loss: 3.4407e-06
Epoch 8/15
247/247 ━━━━━━━━━━━━━━━━━━━━ 156s 630ms/step - loss: 3.4220e-06 - val_loss: 3.4407e-06
Epoch 9/15
247/247 ━━━━━━━━━━━━━━━━━━━━ 154s 622ms/step - loss: 3.4220e-06 - val_loss: 3.4407e-06
Epoch 10/15
247/247 ━━━━━━━━━━━━━━━━━━━━ 155s 629ms/step - loss: 3.4220e-06 - val_loss: 3.4407e-06
Epoch 11/15
247/247 ━━━

In [5]:
# Save full autoencoder
autoencoder.save("D:/PomegranateGuard/model/autoencoder_model.h5")

# Save encoder separately
encoder = Model(input_img, encoded)
encoder.save("D:/PomegranateGuard/model/encoder_model.h5")

print("Autoencoder and Encoder Saved Successfully")

Autoencoder and Encoder Saved Successfully
